# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. All dataset elements such as record sets and fields are always referenced by their `@id` for reproducibility and clarity.

### Dataset Source
The dataset is defined by a Croissant schema and accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We will use the Croissant schema URL to initialize the dataset object and display the core metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Metadata is an object; print key fields
print("Dataset title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", dataset.metadata.version)
print("License:", dataset.metadata.license)
print("Keywords:", dataset.metadata.keywords)
print("Published:", dataset.metadata.date_published)
print("Citation:", dataset.metadata.cite_as)


## 2. Data Overview
Let's review the available record sets, fields, and their IDs. **All references are always via their `@id`.**

In [ ]:
print("Available record set @ids in this dataset:")
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
pprint(record_set_ids)
print()

# For each record set, list the field and column @ids/names
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet @id: {record_set['@id']}")
    print("  Name:", record_set.get('name', ''))
    print("  Description:", record_set.get('description', ''))
    fields = record_set.get('field', [])
    if isinstance(fields, dict):  # single element cast to list
        fields = [fields]
    print("  Fields:")
    for field_ref in fields:
        # field_ref could be dict with '@id'
        fname = field_ref['@id']
        print(f"    @id: {fname}")
        # Let's find the actual field definition
        field_def = next((field for field in dataset.metadata.fields if field['@id'] == fname), None)
        if field_def:
            print("      name:", field_def.get('name', ''))
            print("      description:", field_def.get('description', ''))
            # If the field is mapped directly to a column, print column id
            column = field_def.get('column', None)
            if column:
                # column may be dict or list
                if isinstance(column, list):
                    for c in column:
                        print(f"      column @id: {c['@id']}")
                elif isinstance(column, dict):
                    print(f"      column @id: {column['@id']}")
    print()

## 3. Data Extraction

Load data from each record set into a DataFrame. We will use the record set and field `@id`s identified above.

In [ ]:
# Choose the record set(s) to be loaded. (Adjust the following list based on the output above)
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records from each record set
    print(f"Loading records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}:\n", df.columns.tolist())
    print("First rows:")
    display(df.head(2))
    print("-"*60)
# For demonstration below, pick the first record set loaded
target_record_set_id = record_set_ids[0]
print(f"We will use record set '@id': {target_record_set_id} for EDA.")

## 4. Exploratory Data Analysis (EDA)

We'll choose a numeric field for analysis (referenced by its `@id`). We will perform filtering, normalization, and group-by operations where possible.

In [ ]:
# Let's inspect available fields in our selected record set
df = dataframes[target_record_set_id]
print("Columns in the DataFrame:", df.columns.tolist())

# Pick a numeric field (adjust the column name based on actual data; example uses 'MW_(kDa)' for molecular weight)
candidate_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", candidate_numeric_fields)
# For illustration, pick the first numeric column found; otherwise, set explicitly
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback if types aren't inferred

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
print(f"Using numeric field '{numeric_field_id}', threshold {threshold:.2f}")

# Filtering
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If a categorical/grouping field exists, group by it (e.g., 'Peptide_Count', 'Sample')
candidate_group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < 8]
print("Candidate group-by fields (with <8 uniq values):", candidate_group_fields)
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df)
else:
    print("No suitable grouping field found.")

## 5. Visualization
Let's visualize the distribution of the numeric field and, if relevant, group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id], bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If we did a group mean, show as a bar plot
if 'grouped_df' in locals():
    plt.figure(figsize=(7, 5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion

We demonstrated loading and processing the FAIR^2 mass spectrometry dataset by referencing all record sets and fields via their `@id` fields. With `mlcroissant`, loading the dataset schema, extracting records, and performing exploratory data analysis is unified and reproducible. You can extend this workflow for more advanced analysis or integrate with ML pipelines.